In [1]:
import torch
import torch.nn as nn
from transformers import AutoModelForTokenClassification

d:\ComputerScience\BachKhoa\ProjectII\YOLOQA\venv\lib\site-packages\tqdm\auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cpu


In [3]:
class QASpanDetector(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = AutoModelForTokenClassification.from_pretrained('bert-base-uncased', num_labels=2)

    def forward(self, input_ids, attention_mask):
        outputs = self.model(input_ids=input_ids,
                             attention_mask=attention_mask)
        return outputs

In [4]:
model = QASpanDetector()
model = model.to(device)

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForTokenClassification: ['cls.predictions.decoder.weight', 'cls.predictions.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.seq_relationship.weight', 'cls.predictions.transform.dense.bias', 'cls.seq_relationship.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-u

In [5]:
import pickle

# with open('data/debug_preprocessed2.pkl', 'rb') as f:
with open('data/debug_edge_case.pkl', 'rb') as f:
    train_encodings = pickle.load(f)

In [6]:
len(train_encodings['input_ids'])

754

In [7]:
len(train_encodings['input_ids'][0])

512

In [8]:
len(train_encodings['attention_mask'][2])

512

In [9]:
from preprocess import SquadDataset

train_dataset = SquadDataset(train_encodings)

In [10]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=1, shuffle=False)

In [11]:
def get_batch_answer_length(answer_start, reg_vector):
    reg_vector_temp = []
    for index, start in enumerate(answer_start):
        if start >= 512:
            reg_vector_temp.append(torch.FloatTensor([0]))
        else:
            reg_vector_temp.append(reg_vector[index, int(start)])
    
    reg_vector_temp = torch.FloatTensor(reg_vector_temp).reshape(len(answer_start), 1)
    return reg_vector_temp

In [23]:
import torch.nn.functional as F

def get_loss(reg_pred, obj_pred, reg_target, obj_target, answer_start):
    # Get answer length by get the the value at the answer start index of predict and target vector
    # reg_pred = torch.gather(reg_pred, 1, answer_start).exp()
    # reg_target = torch.gather(reg_target, 1, answer_start)

    batch_size = answer_start.size(0)
    print(reg_pred.requires_grad)
    if int(answer_start) >= 512:
        reg_loss = torch.zeros((1, 1))
    else:
        reg_pred = reg_pred[:, int(answer_start)].exp()
        reg_target = reg_target[:, int(answer_start)]
        print(reg_pred.requires_grad)
        # reg_loss = F.mse_loss(reg_pred, reg_target)
        intersection = torch.minimum(reg_pred, reg_target)
        union = torch.maximum(reg_pred, reg_target)
        print("### IOU Debug ###")
        print(reg_pred)
        print(reg_target)
        iou = intersection / union
        print(intersection)
        print(union)
        print(iou)
        reg_loss = -iou.log().sum() / batch_size
        print("## END Debug")
        
    # reg_pred = get_batch_answer_length(answer_start, reg_pred).to(device)
    # print(reg_pred.requires_grad)
    # reg_pred = reg_pred.exp()
    # reg_target = get_batch_answer_length(answer_start, reg_target)
    # print("Answer Start: ", answer_start)
    # print("Reg Pred:", reg_pred)
    # print("Reg Target:", reg_target)

    
    # reg_loss = F.mse_loss(reg_pred, reg_target)
    obj_loss = F.binary_cross_entropy_with_logits(obj_pred, obj_target, reduction="sum") / batch_size
    print("Reg Loss Backprop:", reg_loss.requires_grad)
    print("Obj Loss Backprop:", obj_loss.requires_grad)

    return reg_loss, obj_loss

In [13]:
MODEL_MAX_LENGTH = 512

In [14]:
import numpy as np

def create_labels(encodings):
    encodings['answer_length'] = np.array(encodings['end_positions'])\
        - np.array(encodings['start_positions']) + 1

    labels = np.zeros((len(encodings['input_ids']), MODEL_MAX_LENGTH, 
        2)) # num_example * seq_length * 2

    for example_idx, start in enumerate(encodings['start_positions']):
        if start < MODEL_MAX_LENGTH: # if the answer is not truncated
            labels[example_idx, start, 0] = 1
            labels[example_idx, start, 1] = encodings['answer_length'][example_idx]

    encodings['labels'] = labels

    return encodings

In [24]:
from torch.optim import AdamW

torch.manual_seed(0)
epochs = 3
print_every = 20
optim = AdamW(model.parameters(), lr=5e-5)
for epoch in range(epochs):
    # Set model in train mode
    model.train()
    loss_of_epoch = 0

    print("############Train############")
    for batch_idx, batch in enumerate(train_loader):
        batch = create_labels(batch)
        sentence_length = batch['input_ids'].size(1)
        batch_size = batch['input_ids'].size(0)

        answer_start = batch['start_positions'].reshape(batch_size, 1).to(device)        
        attention_mask = batch['attention_mask']
        attention_mask = F.pad(attention_mask, (0, MODEL_MAX_LENGTH - sentence_length), 'constant', 0)

        reg_target = batch['labels'][:, :, 1]
        obj_target = batch['labels'][:, :, 0]
        obj_target = torch.FloatTensor(obj_target).to(device)
        reg_target = torch.FloatTensor(reg_target).to(device)

        outputs = model(batch['input_ids'].to(device), batch['attention_mask'].to(device))

        # print(outputs['logits'])
        reg_predict = outputs['logits'][:, :, 1]
        obj_predict = outputs['logits'][:, :, 0]
        reg_predict = F.pad(reg_predict, (0, MODEL_MAX_LENGTH - sentence_length), 'constant', 0).to(device)
        obj_predict = F.pad(obj_predict, (0, MODEL_MAX_LENGTH - sentence_length), 'constant', 0)
        obj_predict = obj_predict * attention_mask.to(device)
        obj_predict = obj_predict.to(device)
        
        reg_loss, obj_loss = get_loss(reg_predict, obj_predict, reg_target, obj_target, answer_start)
        print("Reg Loss:", reg_loss)
        print("Obj Loss:", obj_loss)
        loss = reg_loss + obj_loss
        print("Loss:", loss)
        # loss.backward()
        # optim.step()
        # loss_of_epoch += loss.item()
        # if (batch_idx + 1) % print_every == 0:
        #     print("Batch {:} / {:}".format(batch_idx + 1, len(train_loader)))
        #     print("Loss:", round(loss.item(), 1), "\n")
        if batch_idx == 2:
            break
    break

############Train############
True
Reg Loss Backprop: False
Obj Loss Backprop: True
Reg Loss: tensor([[0.]])
Obj Loss: tensor(393.9807, grad_fn=<DivBackward0>)
Loss: tensor([[393.9807]], grad_fn=<AddBackward0>)
True
True
### IOU Debug ###
tensor([1.8472], grad_fn=<ExpBackward0>)
tensor([4.])
tensor([1.8472], grad_fn=<MinimumBackward0>)
tensor([4.], grad_fn=<MaximumBackward0>)
tensor([0.4618], grad_fn=<DivBackward0>)
## END Debug
Reg Loss Backprop: True
Obj Loss Backprop: True
Reg Loss: tensor(0.7726, grad_fn=<DivBackward0>)
Obj Loss: tensor(378.6094, grad_fn=<DivBackward0>)
Loss: tensor(379.3820, grad_fn=<AddBackward0>)
True
True
### IOU Debug ###
tensor([1.1682], grad_fn=<ExpBackward0>)
tensor([3.])
tensor([1.1682], grad_fn=<MinimumBackward0>)
tensor([3.], grad_fn=<MaximumBackward0>)
tensor([0.3894], grad_fn=<DivBackward0>)
## END Debug
Reg Loss Backprop: True
Obj Loss Backprop: True
Reg Loss: tensor(0.9432, grad_fn=<DivBackward0>)
Obj Loss: tensor(379.1913, grad_fn=<DivBackward0>)
Lo

In [27]:
outputs["logits"].sigmoid()

tensor([[[0.5214, 0.6407],
         [0.4261, 0.5154],
         [0.5866, 0.6093],
         ...,
         [0.4573, 0.5256],
         [0.4623, 0.5949],
         [0.4494, 0.5760]]], grad_fn=<SigmoidBackward0>)